# 🐍 3차시 실습 — API 호출부터 LLM 에이전트까지 (실전 프로젝트)

이 파일은 3차시 Colab 실습용 노트북입니다.  
가상환경 세팅과 파일 분리는 Cursor에서 진행하고, 이 노트북에서는 **2차시 복습 & 문자열 다루기 → Tavily API 호출 → 예외 처리 → LLM API 연동 & 객체 다루기**를 연습합니다.

---
## Part 0. 2차시 복습 — 리스트, 딕셔너리, 중첩 구조, for, if

지난 시간에 배운 핵심을 빠르게 복습합니다.

In [ ]:
# 리스트와 딕셔너리 복습
movies = [
    {"title": "인셉션", "rating": 9.0},
    {"title": "인터스텔라", "rating": 9.5},
    {"title": "다크나이트", "rating": 9.2}
]

# 중첩 구조 파싱
print(movies[1]["title"])  # 인터스텔라

# for + if 조합
for movie in movies:
    if movie["rating"] >= 9.2:
        print(f"{movie['title']} — {movie['rating']}점")

---
## Part 1. 문자열 다루기

LLM API 응답을 정리할 때 자주 쓰는 문자열 메서드를 익혀봅시다.  
API에서 받아온 텍스트에는 불필요한 공백, 개행, 구분자 등이 섞여 있는 경우가 많습니다.

### 1-1. `.strip()` — 앞뒤 공백/개행 제거

In [ ]:
# API 응답에 앞뒤 공백이나 개행이 붙어오는 경우
response = "   파이썬은 범용 프로그래밍 언어입니다.   \n"
print(f"원본: [{response}]")

cleaned = response.strip()
print(f"정리: [{cleaned}]")

In [ ]:
# 개행만 제거하고 싶을 때
text = "첫 번째 줄\n"
print(text.strip("\n"))

### 1-2. `.split()` — 문자열을 나누기

In [ ]:
# 기본: 공백 기준으로 나누기
sentence = "파이썬 자바 자바스크립트"
languages = sentence.split()
print(languages)  # 리스트가 된다!

In [ ]:
# 특정 구분자로 나누기
tags = "AI,데이터,자동화"
tag_list = tags.split(",")
print(tag_list)

In [ ]:
# 여러 줄 텍스트를 줄 단위로 나누기
multi_line = "첫 번째 줄\n두 번째 줄\n세 번째 줄"
lines = multi_line.split("\n")
print(lines)
print(lines[0])  # 첫 번째 줄만

### 1-3. `.join()` — 리스트를 하나의 문자열로 합치기

In [ ]:
# 리스트를 하나의 문자열로 합치기
keywords = ["AI", "데이터", "자동화"]

# 쉼표로 합치기
result = ", ".join(keywords)
print(result)

# 개행으로 합치기 — 여러 검색 결과를 프롬프트에 넣을 때 유용
result = "\n".join(keywords)
print(result)

In [ ]:
# 실전 예시: 검색 결과 제목들을 프롬프트에 넣기
titles = ["파이썬 입문 가이드", "AI 에이전트 만들기", "API 호출 방법"]
prompt = f"""다음 검색 결과를 요약해주세요:

{chr(10).join(titles)}"""
print(prompt)

### 1-4. `.replace()` — 특정 문자열 치환

In [ ]:
# 특정 텍스트를 다른 텍스트로 바꾸기
text = "Python은 어렵다"
fixed = text.replace("어렵다", "재미있다")
print(fixed)

In [ ]:
# 불필요한 문자 제거에도 활용
messy = "[요약] 파이썬은 범용 언어입니다."
clean = messy.replace("[요약] ", "")
print(clean)

In [ ]:
# ✏️ 직접 해보기
# 1. response의 앞뒤 공백을 제거해보세요
response = "  안녕하세요, 무엇을 도와드릴까요?  \n"
print(response)          # 공백 포함


# 2. fruits를 " / "로 합쳐서 출력해보세요
fruits = ["사과", "바나나", "딸기"]
print()                  # "사과 / 바나나 / 딸기"


# 3. text에서 "싫어"를 "좋아"로 바꿔보세요
text = "나는 코딩이 싫어"
print()                  # "나는 코딩이 좋아"

---
## Part 2. API란 무엇인가?

API(Application Programming Interface)는 **프로그램끼리 대화하는 방법**입니다.  
우리가 브라우저로 검색할 때는 사람이 직접 검색어를 입력하고 결과를 눈으로 봅니다.  
하지만 파이썬 코드에서는 API를 통해 **자동으로 요청을 보내고 결과를 받아올** 수 있습니다.

### 요청과 응답의 흐름

```
내 코드 (요청) →→→ API 서버 →→→ 내 코드 (응답 받음)
  "AI 뉴스 검색해줘"           {결과 JSON 데이터}
```

### API 키

대부분의 API는 **API 키**(일종의 비밀번호)를 발급받아야 사용할 수 있습니다.  
이 키로 "누가 얼마나 요청했는지"를 추적하고, 악용을 방지합니다.

---
## Part 3. Tavily Search API 호출하기

Tavily는 AI 에이전트를 위한 검색 엔진입니다.  
검색어를 보내면 관련 기사의 제목, URL, 요약 내용을 JSON으로 돌려줍니다.

### 준비물
1. Tavily API 키 발급: [https://tavily.com](https://tavily.com) 에서 회원가입 후 API 키 복사
2. 아래 셀에서 `requests` 패키지 설치

In [ ]:
# Colab에서 패키지 설치
!pip install requests

In [ ]:
# API 키 입력 (Colab에서 테스트용으로 직접 입력)
# 실제 프로젝트에서는 .env 파일에 저장하고 불러옵니다 (Cursor에서 실습)
TAVILY_API_KEY = ""  # 여기에 발급받은 API 키를 붙여넣으세요

### 3-1. 첫 번째 API 호출

In [ ]:
import requests

# Tavily API에 검색 요청 보내기
url = "https://api.tavily.com/search"

# 요청에 보낼 데이터 — 딕셔너리 형태!
data = {
    "api_key": TAVILY_API_KEY,
    "query": "인공지능 최신 뉴스",
    "max_results": 3
}

# POST 요청 보내기
response = requests.post(url, json=data)

# 응답 상태 확인 (200이면 성공)
print(f"상태 코드: {response.status_code}")

In [ ]:
# 응답 데이터를 딕셔너리로 변환
result = response.json()

# 전체 구조 확인 — 어떤 키가 있는지 보기
print(result.keys())

In [ ]:
# 전체 응답을 보기 좋게 출력
import json

print(json.dumps(result, indent=2, ensure_ascii=False))

### 3-2. 응답 구조 파악하기

Tavily API 응답은 2차시에서 연습한 **중첩 구조**와 같은 형태입니다.  
바깥부터 한 단계씩 접근해봅시다.

In [ ]:
# 1단계: results 키로 접근 → 리스트가 나옴
results = result["results"]
print(f"검색 결과 수: {len(results)}개")
print(type(results))  # list

In [ ]:
# 2단계: 첫 번째 결과 꺼내기 → 딕셔너리가 나옴
first = results[0]
print(first.keys())

In [ ]:
# 3단계: 제목과 내용 꺼내기
print(f"제목: {first['title']}")
print(f"URL: {first['url']}")
print(f"내용: {first['content'][:100]}...")  # 앞 100자만

In [ ]:
# 한 줄로 연결해서 쓰기 (2차시에서 배운 방식)
print(result["results"][0]["title"])

### 3-3. `for`문으로 모든 결과 파싱하기

In [ ]:
# 모든 검색 결과의 제목과 요약 출력
for i, item in enumerate(result["results"]):
    print(f"[{i+1}] {item['title']}")
    print(f"    {item['content'][:80]}...")
    print()

In [ ]:
# 제목만 리스트로 모으기
titles = []
for item in result["results"]:
    titles.append(item["title"])

print(titles)

In [ ]:
# 문자열 메서드 활용: 제목 리스트를 하나의 텍스트로 합치기
titles_text = "\n".join(titles)
print(titles_text)

In [ ]:
# ✏️ 직접 해보기
# 모든 검색 결과에서 content만 모아서 리스트로 만들어보세요

contents = []
for item in result[""]:
    contents.append(item[""])  # 어떤 키를 넣어야 할까요?

# 내용을 번호 붙여서 출력
for i, content in enumerate(contents):
    print(f"[{i+1}] {content[:80]}...")
    print()

### 3-4. 다른 검색어로 호출해보기

In [ ]:
# 검색어를 바꿔서 다시 호출
data = {
    "api_key": TAVILY_API_KEY,
    "query": "파이썬 자동화",  # 검색어를 바꿔보세요
    "max_results": 5
}

response = requests.post(url, json=data)
result = response.json()

for i, item in enumerate(result["results"]):
    print(f"[{i+1}] {item['title']}")
    print(f"    {item['content'][:80]}...")
    print()

In [ ]:
# ✏️ 직접 해보기
# 원하는 검색어로 Tavily API를 호출하고, 결과를 출력해보세요

data = {
    "api_key": TAVILY_API_KEY,
    "query": "",        # 원하는 검색어를 넣어보세요
    "max_results": 3
}

response = requests.post(url, json=data)
result = response.json()

for i, item in enumerate(result["results"]):
    print(f"[{i+1}] {item['title']}")

### 3-5. API 호출을 함수로 만들기

같은 코드를 매번 복사하지 말고, 함수로 만들어두면 재사용이 편합니다.  
이 함수를 나중에 Cursor에서 `search.py` 파일로 옮기게 됩니다.

In [ ]:
def search_tavily(query, max_results=3):
    """Tavily API로 검색하고 결과 리스트를 반환합니다."""
    url = "https://api.tavily.com/search"
    data = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results
    }
    response = requests.post(url, json=data)
    result = response.json()
    return result["results"]

# 함수로 호출
results = search_tavily("AI 에이전트")

for i, item in enumerate(results):
    print(f"[{i+1}] {item['title']}")

In [ ]:
# 검색어만 바꾸면 바로 재사용 가능
results = search_tavily("LLM 활용 사례", max_results=5)

for i, item in enumerate(results):
    print(f"[{i+1}] {item['title']}")
    print(f"    {item['content'][:80]}...")
    print()

---
## Part 4. 예외 처리 (`try-except`)

API를 호출할 때는 다양한 이유로 실패할 수 있습니다.
- 인터넷 연결이 끊김
- API 키가 잘못됨
- 서버가 일시적으로 응답하지 않음

`try-except`를 사용하면 에러가 나도 **프로그램이 죽지 않고** 대응할 수 있습니다.

In [ ]:
# try-except 기본 구조
try:
    # 에러가 날 수 있는 코드
    result = 10 / 0
except:
    # 에러가 났을 때 실행되는 코드
    print("에러가 발생했습니다!")

In [ ]:
# try-except가 없으면 프로그램이 여기서 멈춤
result = 10 / 0  # ZeroDivisionError
print("이 줄은 실행되지 않습니다")

In [ ]:
# try-except가 있으면 에러를 잡고 다음 코드를 계속 실행
try:
    result = 10 / 0
except:
    print("0으로 나눌 수 없습니다")

print("프로그램이 계속 실행됩니다!")

In [ ]:
# 에러 메시지를 확인하기 — as e로 에러 내용을 변수에 담기
try:
    result = 10 / 0
except Exception as e:
    print(f"에러 종류: {type(e).__name__}")
    print(f"에러 내용: {e}")

### 4-1. API 호출에 예외 처리 적용하기

In [ ]:
# 잘못된 API 키로 호출하면?
data = {
    "api_key": "잘못된_키",
    "query": "테스트",
    "max_results": 3
}

try:
    response = requests.post(url, json=data)
    result = response.json()
    print(result)
except Exception as e:
    print(f"API 호출 실패: {e}")

In [ ]:
# 검색 함수에 예외 처리 추가하기
def search_tavily_safe(query, max_results=3):
    """예외 처리가 포함된 Tavily 검색 함수"""
    url = "https://api.tavily.com/search"
    data = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results
    }

    try:
        response = requests.post(url, json=data)
        result = response.json()

        if "results" in result:
            return result["results"]
        else:
            print(f"예상치 못한 응답: {result}")
            return []
    except Exception as e:
        print(f"검색 실패: {e}")
        return []

# 정상 호출
results = search_tavily_safe("AI 에이전트")
print(f"검색 결과 {len(results)}건")

for item in results:
    print(f"- {item['title']}")

In [ ]:
# ✏️ 직접 해보기
# 아래 코드에 try-except를 추가해서 에러가 나도 프로그램이 안 죽게 만들어보세요

numbers = [10, 0, 5]

for num in numbers:
    result = 100 / num  # num이 0일 때 에러!
    print(f"100 / {num} = {result}")

---
## Part 5. LLM API 호출과 객체(Object) 다루기

LLM API를 호출하면 응답이 딕셔너리가 아니라 **객체(Object)**로 옵니다.  
여기서는 객체가 뭔지, 딕셔너리와 어떻게 다른지를 단계별로 알아봅시다.

> 패키지 설치와 API 키 세팅은 **Cursor에서 진행**합니다.  
> 여기서는 더미 클래스로 객체 다루는 방법을 먼저 연습합니다.

### 5-1. 딕셔너리 방식 복습

지금까지는 데이터를 **딕셔너리**에 담고, `["키"]`로 꺼냈습니다.

In [ ]:
# 딕셔너리로 데이터 담기
dict_response = {"content": "파이썬은 범용 언어입니다.", "model": "gpt-4o-mini"}

# 대괄호 + 키 이름으로 꺼내기
print(dict_response["content"])
print(dict_response["model"])

### 5-2. 객체(Object)란?

객체도 데이터를 담는 그릇이지만, 꺼내는 방법이 다릅니다.

| 구분 | 꺼내는 방법 | 예시 |
|------|------------|------|
| **딕셔너리** | `["키"]` | `result["content"]` |
| **객체** | `.속성이름` | `response.content` |

점(`.`)으로 꺼낸다는 것만 기억하면 됩니다.  
사실 `.append()`, `.keys()` 같은 메서드도 점(`.`)으로 접근한 것이니, 이미 쓰고 있던 패턴입니다.

In [ ]:
# 연습용 객체를 만들기 위한 클래스 정의
# 지금은 "이런 틀이 있구나" 정도로만 이해하면 됩니다
class AIMessage:
    def __init__(self, content, model):
        self.content = content
        self.model = model

In [ ]:
# 객체 만들기
obj_response = AIMessage(content="파이썬은 범용 언어입니다.", model="gpt-4o-mini")

# 점(.)으로 값 꺼내기
print(obj_response.content)
print(obj_response.model)

### 5-3. 딕셔너리 vs 객체 — 나란히 비교

In [ ]:
# 같은 데이터를 딕셔너리와 객체로 각각 꺼내보기

# 딕셔너리: ["키"]로 꺼냄
dict_response = {"content": "파이썬은 범용 언어입니다.", "model": "gpt-4o-mini"}
print("딕셔너리:", dict_response["content"])

# 객체: .속성으로 꺼냄
obj_response = AIMessage(content="파이썬은 범용 언어입니다.", model="gpt-4o-mini")
print("객체:    ", obj_response.content)

### 5-4. LLM API 응답이 객체인 이유

LangChain 같은 라이브러리는 LLM 응답을 `AIMessage`라는 객체로 돌려줍니다.  
실제 코드는 이렇게 생겼습니다 (Cursor에서 실행할 예정):

In [ ]:
# 실제 LLM 호출 코드 (지금은 주석으로만 확인)
# Cursor에서 패키지 설치 후 실행합니다

# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")
# response = llm.invoke("파이썬이 뭐야?")

# 응답 타입 확인
# print(type(response))    # <class 'AIMessage'> ← 딕셔너리가 아님!

# 객체니까 점(.)으로 꺼냄
# print(response.content)  # "파이썬은 범용 프로그래밍 언어입니다..."

# 주의: 딕셔너리처럼 쓰면 에러!
# print(response["content"])  # ← TypeError 발생

### 5-5. 객체에서 꺼낸 값에 문자열 메서드 쓰기

`.content`로 꺼낸 값은 문자열이므로, Part 1에서 배운 `.strip()`, `.split()` 등을 바로 연결할 수 있습니다.

In [ ]:
obj_response = AIMessage(
    content="AI 에이전트는 자동으로 작업을 수행하는 프로그램입니다.",
    model="gpt-4o-mini"
)

# .content로 텍스트 꺼내기
text = obj_response.content
print(text)

In [ ]:
# 꺼낸 텍스트에 문자열 메서드 적용하기
words = obj_response.content.split()
print(f"단어 수: {len(words)}개")
print(f"단어 목록: {words}")

In [ ]:
# ✏️ 직접 해보기 1
# obj_response에서 model 속성을 꺼내서 출력해보세요
obj_response = AIMessage(content="LLM은 대형 언어 모델입니다.", model="gpt-4o-mini")

print(obj_response.)    # 점 뒤에 어떤 속성 이름을 쓰면 될까요?

In [ ]:
# ✏️ 직접 해보기 2
# obj_response.content에서 "대형"이라는 단어가 포함되어 있는지 if문으로 확인해보세요
obj_response = AIMessage(content="LLM은 대형 언어 모델입니다.", model="gpt-4o-mini")

if "" in obj_response.:    # 빈칸을 채워보세요
    print("'대형'이 포함되어 있습니다")

---
## Part 6. 에이전트 조립 미리보기

3차시의 나머지 실습은 **Cursor에서 진행**합니다.  
아래는 최종 에이전트의 전체 흐름을 미리 보는 코드입니다.

### 파일 구조
```
my-agent/
├── .env           ← API 키 보관
├── main.py        ← 실행 (검색 → 요약 → 저장)
├── search.py      ← Tavily 검색 함수
├── llm.py         ← LLM 요약 함수
└── requirements.txt
```

In [ ]:
# 에이전트 전체 흐름 미리보기 (의사 코드)
# 실제 실행은 Cursor에서 합니다

# === search.py ===
# def search_tavily(query):
#     ... API 호출 ...
#     return results  # 기사 리스트

# === llm.py ===
# def summarize(text):
#     ... LLM 호출 ...
#     return response.content  # 요약 텍스트

# === main.py ===
# from search import search_tavily
# from llm import summarize
#
# articles = search_tavily("AI 최신 뉴스")
#
# summaries = []
# for article in articles:
#     summary = summarize(article["content"])
#     summaries.append({
#         "title": article["title"],
#         "summary": summary
#     })
#
# # JSON 파일로 저장
# import json
# with open("result.json", "w") as f:
#     json.dump(summaries, f, ensure_ascii=False, indent=2)

print("이 흐름을 Cursor에서 직접 만들어봅시다!")